# 🤗 Défi quotidien : Système RAG avec LangChain & Hugging Face

Nous allons construire un système de **Retrieval-Augmented Generation (RAG)** complet :
1. Charger le dataset `databricks/databricks-dolly-15k`
2. Découper, embedder et indexer les documents dans FAISS
3. Brancher un LLM Hugging Face
4. Répondre à des questions grâce à la chaîne `RetrievalQA`

---
## 1) Configuration de l'environnement

In [ ]:
# Installer toutes les bibliothèques nécessaires
# Chaque lib a un rôle précis dans le pipeline RAG :
# - langchain          : orchestration des composants
# - torch              : backend de calcul pour les modèles
# - transformers       : modèles pré-entraînés Hugging Face
# - sentence-transformers : génération des embeddings sémantiques
# - datasets           : chargement des datasets HF
# - faiss-cpu          : recherche de similarité vectorielle rapide
# - langchain-community: loaders, vectorstores, pipelines HF pour LangChain

!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -Uq datasets
!pip install -q faiss-cpu
!pip install -U langchain-community

print("✅ Installation terminée.")

---
## 2) Chargement du dataset

On utilise `HuggingFaceDatasetLoader` pour charger `databricks/databricks-dolly-15k` directement depuis le Hub Hugging Face et le convertir en documents LangChain.

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Nom du dataset et colonne contenant le texte principal
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"  # La colonne "context" contient le texte de référence

# Charger les données sous forme de documents LangChain
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

print(f"✅ {len(data)} documents chargés.")
print("\nAperçu des 2 premiers documents :")
print(data[:2])

---
## 3) Découpage des documents en chunks

Les LLMs ont une fenêtre de contexte limitée. On découpe chaque document en segments plus petits avec un léger chevauchement (`chunk_overlap`) pour ne pas perdre d'information aux frontières.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=1000 : chaque chunk fait au plus 1000 caractères
# chunk_overlap=150 : 150 caractères partagés entre chunks consécutifs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)

print(f"✅ {len(docs)} chunks créés à partir de {len(data)} documents.")
print("\nPremier chunk :")
print(docs[0])

---
## 4) Génération des embeddings

On convertit chaque chunk en vecteur numérique dense avec `all-MiniLM-L6-v2`, un modèle léger qui capture le sens sémantique des textes.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Modèle d'embedding : rapide, léger, 384 dimensions
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device': 'cpu'}         # Utiliser le CPU (pas de GPU requis)
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test optionnel : vérifier que les embeddings fonctionnent
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(f"✅ Embeddings prêts. Dimension : {len(query_result)}")
print(f"Premiers 3 valeurs du vecteur test : {query_result[:3]}")

---
## 5) Création du vectorstore FAISS

FAISS indexe tous les vecteurs pour permettre des recherches de similarité très rapides (même sur des millions de vecteurs).

⚠️ Cette étape peut prendre quelques minutes selon la taille du dataset.

In [ ]:
from langchain_community.vectorstores import FAISS

# Indexer tous les chunks avec leurs embeddings
db = FAISS.from_documents(docs, embeddings)

print("✅ Vectorstore FAISS créé.")
print(f"   Nombre de vecteurs indexés : {db.index.ntotal}")

---
## 6) Préparation du LLM (modèle de questions-réponses)

On utilise `Intel/dynamic_tinybert`, un modèle léger spécialisé en question-answering (format extractif : il extrait la réponse depuis un contexte donné).

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

# Charger le tokenizer avec troncature pour éviter les erreurs de longueur
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding=True,
    truncation=True,
    max_length=512
)

# Pipeline HuggingFace pour la question-answering extractive
# return_tensors='pt' : retourner des tenseurs PyTorch
qa_pipeline = pipeline(
    "question-answering",
    model=model_name,
    tokenizer=tokenizer,
    return_tensors='pt'
)

# Wrapper LangChain autour du pipeline HuggingFace
llm = HuggingFacePipeline(
    pipeline=qa_pipeline,
    model_kwargs={"temperature": 0.7, "max_length": 512},
)

print("✅ LLM (Intel/dynamic_tinybert) prêt.")

---
## 7) Création de la chaîne RetrievalQA

On relie le retriever (qui cherche les documents pertinents) au LLM (qui génère la réponse). Le flux complet est :

**Question → FAISS retriever → k documents → LLM → Réponse**

In [ ]:
from langchain.chains import RetrievalQA

# Créer le retriever depuis le vectorstore FAISS
# k=4 : récupérer les 4 chunks les plus similaires à la question
retriever = db.as_retriever(search_kwargs={"k": 4})

# Construire la chaîne RetrievalQA
# chain_type="refine" : raffine itérativement la réponse avec chaque document récupéré
# return_source_documents=False : ne pas retourner les docs sources dans le résultat
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False
)

print("✅ Chaîne RetrievalQA prête.")

---
## 8) Test du système RAG

On soumet une question au système et on observe la réponse générée à partir des documents récupérés.

In [ ]:
# Question de test
question = "What is cheesemaking?"

# Exécuter la chaîne RAG
result = qa.run({"query": question})

print("=" * 60)
print(f"Q: {question}")
print("=" * 60)
print(f"A: {result}")

In [ ]:
# Tester avec d'autres questions pour explorer le système
questions = [
    "What is the capital of France?",
    "How does photosynthesis work?",
    "What are the main ingredients in bread?"
]

for q in questions:
    print("\n" + "=" * 60)
    print(f"Q: {q}")
    try:
        answer = qa.run({"query": q})
        print(f"A: {answer}")
    except Exception as e:
        print(f"⚠️ Erreur : {e}")

---
## ✅ Récapitulatif — Ce que vous avez construit

| Étape | Outil | Rôle |
|-------|-------|------|
| Chargement | `HuggingFaceDatasetLoader` | Importer `databricks-dolly-15k` |
| Découpage | `RecursiveCharacterTextSplitter` | Chunks de 1000 car., overlap 150 |
| Embeddings | `all-MiniLM-L6-v2` | Représentations vectorielles sémantiques |
| Vectorstore | FAISS | Index pour recherche de similarité rapide |
| LLM | `Intel/dynamic_tinybert` | Extraction de réponses |
| Pipeline RAG | `RetrievalQA` | Retrieve → Refine → Answer |

**💡 Pistes d'amélioration :**
- Augmenter `k` dans le retriever pour plus de contexte
- Tester `chain_type="stuff"` vs `"refine"` vs `"map_reduce"`
- Remplacer `dynamic_tinybert` par un modèle génératif plus puissant (ex. `flan-t5-base`)
- Activer `return_source_documents=True` pour voir quels passages ont été utilisés